# Quantum Computing Essentials with Qiskit

**Qiskit Fall Fest workshop notebook.** Written for Qiskit 2.x, runs in Google Colab, no account needed until the last chapters.

---

## How to read this notebook

Think of this as a short book that happens to run. Each chapter follows the same rhythm:

1. We **look** at something in code (a circuit drawing, a Bloch sphere, a histogram).
2. We **explain** the mathematics behind what we just saw.
3. We **stop and ask you to try** something yourself.

A few conventions are used throughout:

> **Predict.** Before running a cell, write down what you expect to see. Being wrong is the useful part.

> **Physics note.** A short historical or physical aside that connects the code to the real laboratory.

> **Check.** A cell that tests your own answer and tells you whether it is right.

Exercises have hints and full solutions hidden inside collapsible blocks. Open them only after a real attempt.

## Contents

| Chapter | Topic |
|---|---|
| 0 | Setting up |
| 1 | Qubits |
| 2 | The Bloch sphere |
| 3 | Gates |
| 4 | Measurement |
| 5 | Your first circuit |
| 6 | Entanglement |
| 7 | Interference |
| 8 | Noise and real devices |
| 9 | Running on IBM Quantum hardware |
| 10 | A worked problem: adding two bits |
| | Where we go next, and further reading |

If you want to go deeper on any chapter, each one ends with a link to the matching page in the IBM Quantum documentation.

## Chapter 0. Setting up

We use three packages:

- `qiskit`, the core SDK for building circuits, plus visualisation and reference simulators.
- `qiskit-aer`, a fast simulator that can also model noise.
- `qiskit-ibm-runtime`, the client for IBM's cloud hardware. It also ships snapshots of real devices that we can simulate offline.

Run the next cell once. It takes about a minute on a fresh Colab session.

In [ ]:
%pip install -q "qiskit[visualization]" qiskit-aer qiskit-ibm-runtime pylatexenc ipywidgets

In [ ]:
import io
import numpy as np
import matplotlib.pyplot as plt

import qiskit
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector, Operator, SparsePauliOp, Pauli
from qiskit.visualization import (plot_bloch_vector, plot_bloch_multivector,
                                  plot_histogram)
from qiskit.primitives import StatevectorSampler, StatevectorEstimator
from ipywidgets import interact, FloatSlider

print("Qiskit version:", qiskit.__version__)


# Two small helpers used throughout the notebook.

def fig_to_img(fig):
    # Render a matplotlib Figure to an image array so we can place
    # several Bloch spheres side by side.
    buf = io.BytesIO()
    fig.savefig(buf, format="png", bbox_inches="tight", dpi=100)
    buf.seek(0)
    return plt.imread(buf)


def check(label, test):
    # Run a test (a function returning True/False) and report the result.
    # If your variable is still None or the code raises, we say "Not yet".
    try:
        ok = bool(test())
    except Exception:
        ok = False
    print(("Correct:  " if ok else "Not yet:  ") + label)

## Chapter 1. Qubits

A classical bit is either 0 or 1. A **qubit** is a two-level quantum system whose state is a vector of two complex numbers:

$$|\psi\rangle = \alpha|0\rangle + \beta|1\rangle, \qquad |\alpha|^2 + |\beta|^2 = 1.$$

In column-vector form,

$$|0\rangle = \begin{pmatrix}1\\0\end{pmatrix},\qquad |1\rangle = \begin{pmatrix}0\\1\end{pmatrix},\qquad |\psi\rangle = \begin{pmatrix}\alpha\\ \beta\end{pmatrix}.$$

The numbers $\alpha$ and $\beta$ are called **amplitudes**. Unlike probabilities they may be negative or complex. What we observe in the end are probabilities, given by the **Born rule**: a measurement returns 0 with probability $|\alpha|^2$ and 1 with probability $|\beta|^2$.

> **Physics note.** The prototype of a two-level system is the electron spin. In the Stern-Gerlach experiment (1922) a beam of silver atoms passing through an inhomogeneous magnetic field split into exactly two beams, not a smear. That discreteness is what "two-level" means physically. The Born rule came in 1926, and the bra-ket notation we use here was introduced by Dirac. IBM's qubits are not spins. They are tiny superconducting circuits (transmons) whose two lowest energy levels play the role of $|0\rangle$ and $|1\rangle$, but the mathematics is identical.

In Qiskit, the `Statevector` class stores exactly the vector above.

In [ ]:
zero = Statevector.from_label("0")
one  = Statevector.from_label("1")

print("|0> =", zero.data)
print("|1> =", one.data)

Now a state that is neither $|0\rangle$ nor $|1\rangle$. We choose $\alpha=\sqrt{0.7}$ and $\beta=\sqrt{0.3}$, so we expect to measure 0 about 70% of the time.

In [ ]:
psi = Statevector([np.sqrt(0.7), np.sqrt(0.3)])

display(psi.draw("latex"))                      # the state in Dirac notation
print("Probabilities:", psi.probabilities())    # [P(0), P(1)]

**Normalisation matters.** Probabilities must add up to 1, so not every pair of numbers is a legal state. Qiskit can check this for us.

In [ ]:
bad  = Statevector([1, 1])
good = Statevector(np.array([1, 1]) / np.sqrt(2))

print("Is [1, 1] a valid state?          ", bad.is_valid())
print("Is [1, 1]/sqrt(2) a valid state?  ", good.is_valid())

### Named states

Some states appear so often that Qiskit gives them labels.

| Label | State | Basis |
|---|---|---|
| `"0"`, `"1"` | $\lvert 0\rangle,\ \lvert 1\rangle$ | Z |
| `"+"`, `"-"` | $(\lvert 0\rangle \pm \lvert 1\rangle)/\sqrt2$ | X |
| `"r"`, `"l"` | $(\lvert 0\rangle \pm i\lvert 1\rangle)/\sqrt2$ | Y |

In [ ]:
for lab in ["0", "1", "+", "-", "r", "l"]:
    sv = Statevector.from_label(lab)
    print(f"|{lab}>   amplitudes = {np.round(sv.data, 3)}   P(0), P(1) = {np.round(sv.probabilities(), 3)}")

> **Predict.** Look at the rows for `"+"` and `"-"`. They have identical measurement probabilities. If a measurement cannot tell them apart, in what sense are they different states? Hold on to that question. Chapter 7 answers it.

### Exercise 1: building a state

**Goal.** Create a `Statevector` called `ex1_state` for which $P(0)=0.25$ and $P(1)=0.75$.

<details>
<summary>Hint</summary>

The probabilities are the squares of the amplitudes, so take square roots. Do you need to worry about normalisation?
</details>

In [ ]:
ex1_state = None      # replace None with your Statevector([ ..., ... ])

In [ ]:
check("P(0) = 0.25 and P(1) = 0.75",
      lambda: np.allclose(ex1_state.probabilities(), [0.25, 0.75]))

<details>
<summary>Solution</summary>

```python
ex1_state = Statevector([np.sqrt(0.25), np.sqrt(0.75)])   # = [0.5, 0.866...]
```

Note that `Statevector([-0.5, np.sqrt(0.75)])` or even one with a complex phase on $\beta$ also passes the check. Different states, same probabilities. That is the same phenomenon as with `"+"` and `"-"`.
</details>

**Stretch.** In Chapter 2 we will describe every qubit state by two angles. For this state, can you guess the polar angle $\theta$ using $P(0)=\cos^2(\theta/2)$? Come back to it after Chapter 2.

Further reading: [Qiskit quantum information tools](https://quantum.cloud.ibm.com/docs/en/api/qiskit/quantum_info) and the IBM Quantum Learning course [Basics of quantum information](https://quantum.cloud.ibm.com/learning/en/courses/basics-of-quantum-information).

## Chapter 2. The Bloch sphere

Two facts let us draw a qubit as a point in 3D. First, $|\alpha|^2+|\beta|^2=1$ removes one real degree of freedom. Second, multiplying the whole state by a phase $e^{i\gamma}$ changes nothing observable, which removes another. What remains is two real numbers, and every pure qubit state can be written as

$$|\psi\rangle = \cos\tfrac{\theta}{2}\,|0\rangle + e^{i\varphi}\sin\tfrac{\theta}{2}\,|1\rangle, \qquad 0\le\theta\le\pi,\quad 0\le\varphi<2\pi .$$

These are ordinary spherical angles. The corresponding point on the unit sphere is

$$(x,y,z) = (\sin\theta\cos\varphi,\ \sin\theta\sin\varphi,\ \cos\theta) = \big(\langle X\rangle,\ \langle Y\rangle,\ \langle Z\rangle\big).$$

So the north pole is $|0\rangle$, the south pole is $|1\rangle$, the $\pm x$ axis holds $|\pm\rangle$, and the $\pm y$ axis holds the Y-basis states `"r"` and `"l"`.

> **Physics note.** Felix Bloch introduced this picture in 1946 while describing nuclear magnetic resonance. The vector is literally the average spin direction of an ensemble in a magnetic field. The same sphere appears in optics as the Poincare sphere, where it describes light polarisation. Whenever a physical system has two levels, this geometry comes along for free.

Let us look at some states.

In [ ]:
plot_bloch_vector([0, 0, 1], title="|0>: north pole")

In [ ]:
for lab in ["0", "1", "+", "-", "r", "l"]:
    display(plot_bloch_multivector(Statevector.from_label(lab), title=f"|{lab}>"))

The coordinates are expectation values, so we can compute them for any state. This helper will be reused.

In [ ]:
def bloch_xyz(sv):
    return [sv.expectation_value(Pauli(p)).real for p in "XYZ"]

psi = Statevector([np.sqrt(0.7), np.sqrt(0.3)])
print("Bloch coordinates (x, y, z):", np.round(bloch_xyz(psi), 3))
plot_bloch_multivector(psi, title="sqrt(0.7)|0> + sqrt(0.3)|1>")

### Interactive: move the state yourself

Drag the sliders. $\theta$ is the polar angle measured from the north pole, $\varphi$ is the azimuth around the vertical axis. Watch how the amplitudes and the probabilities respond, and in particular notice that $\varphi$ moves the point without changing $P(0)$ or $P(1)$.

(We reach the state with an $R_y(\theta)$ gate followed by a phase gate $P(\varphi)$. Both are introduced in Chapter 3.)

In [ ]:
@interact(theta=FloatSlider(min=0, max=np.pi, step=0.05, value=np.pi/3, description="theta"),
          phi=FloatSlider(min=0, max=2*np.pi, step=0.05, value=0.0, description="phi"))
def explore(theta, phi):
    qc = QuantumCircuit(1)
    qc.ry(theta, 0)
    qc.p(phi, 0)
    sv = Statevector(qc)
    print("amplitudes:", np.round(sv.data, 3))
    print("P(0), P(1):", np.round(sv.probabilities(), 3))
    fig = plot_bloch_multivector(sv)
    display(fig)
    plt.close(fig)

### Exercise 2: reading the sphere

**Goal.** Find the polar angle `ex2_theta` for which $P(0) = 0.25$, using $P(0)=\cos^2(\theta/2)$. Then check your answer by building the state with `ry` and looking at it on the sphere. This is the stretch question from Exercise 1.

<details>
<summary>Hint</summary>

$\cos(\theta/2) = \sqrt{P(0)} = 0.5$. Which angle has cosine 0.5?
</details>

In [ ]:
ex2_theta = None      # your angle in radians

In [ ]:
def _ex2():
    qc = QuantumCircuit(1); qc.ry(ex2_theta, 0)
    return np.isclose(Statevector(qc).probabilities()[0], 0.25)

check("Ry(theta)|0> has P(0) = 0.25", _ex2)

<details>
<summary>Solution</summary>

$\theta/2 = \arccos(0.5) = \pi/3$, so $\theta = 2\pi/3$.

```python
ex2_theta = 2*np.pi/3
```

On the sphere the point sits $120^\circ$ from the north pole, in the lower hemisphere's direction of $+x$. Since $z=\cos\theta = -0.5$, the state is closer to $|1\rangle$ than to $|0\rangle$, as expected for $P(1)=0.75$.
</details>

Further reading: [Visualize circuits and states](https://quantum.cloud.ibm.com/docs/en/guides/visualize-circuits).

## Chapter 3. Gates

A quantum gate is a **unitary** matrix $U$, meaning $U^\dagger U = I$. It maps states to states, $|\psi'\rangle = U|\psi\rangle$, and unitarity guarantees that the total probability stays 1. On the Bloch sphere every single-qubit gate is a **rotation**.

| Gate | Matrix | Action on the sphere |
|---|---|---|
| X | $\begin{pmatrix}0&1\\1&0\end{pmatrix}$ | half-turn about x |
| Y | $\begin{pmatrix}0&-i\\i&0\end{pmatrix}$ | half-turn about y |
| Z | $\begin{pmatrix}1&0\\0&-1\end{pmatrix}$ | half-turn about z |
| H | $\tfrac{1}{\sqrt2}\begin{pmatrix}1&1\\1&-1\end{pmatrix}$ | half-turn about the axis halfway between x and z |
| S | $\begin{pmatrix}1&0\\0&i\end{pmatrix}$ | quarter-turn about z |
| T | $\begin{pmatrix}1&0\\0&e^{i\pi/4}\end{pmatrix}$ | eighth-turn about z |
| $R_x(\theta),R_y(\theta),R_z(\theta)$ | $e^{-i\theta P/2}$ | rotation by $\theta$ about the chosen axis |

> **Physics note.** X, Y and Z, the Pauli matrices, were written down by Pauli in 1927 to describe electron spin. A rotation $R_z(\theta)$ is what a spin does on its own in a magnetic field along z: it precesses at the Larmor frequency. On a superconducting chip, $R_x$ and $R_y$ rotations are produced by short microwave pulses that drive Rabi oscillations between $|0\rangle$ and $|1\rangle$. When you write `qc.rx(theta, 0)` you are specifying how long and how strongly to drive.

First, the actual matrices Qiskit uses.

In [ ]:
from qiskit.circuit.library import XGate, YGate, ZGate, HGate, SGate, TGate

for name, g in [("X", XGate()), ("Y", YGate()), ("Z", ZGate()),
                ("H", HGate()), ("S", SGate()), ("T", TGate())]:
    print(f"--- {name} ---")
    print(np.round(Operator(g).data, 3))

### Before and after

The helper below prepares a qubit, applies one gate, and shows the state **before** and **after** on the Bloch sphere, along with the one-gate circuit.

In [ ]:
def before_after(gate_name, init=None, args=()):
    # gate_name : any single-qubit QuantumCircuit method name, e.g. "x", "h", "rx"
    # init      : optional QuantumCircuit that prepares the starting state
    # args      : gate parameters, e.g. (np.pi/3,) for rx
    prep = init if init is not None else QuantumCircuit(1)
    full = prep.copy()
    getattr(full, gate_name)(*args, 0)

    before, after = Statevector(prep), Statevector(full)
    print(f"Gate: {gate_name.upper()}{args if args else ''}")
    print("  before:", np.round(before.data, 3))
    print("  after: ", np.round(after.data, 3))

    gate_only = QuantumCircuit(1)
    getattr(gate_only, gate_name)(*args, 0)
    display(gate_only.draw("mpl", scale=0.8))

    fig, axes = plt.subplots(1, 2, figsize=(8, 4))
    for i, (title, sv) in enumerate([("BEFORE", before), ("AFTER", after)]):
        axes[i].axis("off")
        f = plot_bloch_multivector(sv, title=title)
        axes[i].imshow(fig_to_img(f))
        plt.close(f)
    plt.show()

**The X gate** is the quantum NOT: $X|0\rangle = |1\rangle$. The arrow flips from the north pole to the south pole.

In [ ]:
before_after("x")

**The Hadamard gate** turns a definite state into an equal superposition: $H|0\rangle = (|0\rangle+|1\rangle)/\sqrt2 = |+\rangle$. The arrow moves from the pole to the equator, along $+x$.

In [ ]:
before_after("h")

**The Z gate** flips the sign of the $|1\rangle$ amplitude. Acting on $|0\rangle$ or $|1\rangle$ it does nothing visible, so to see its effect we start from $|+\rangle$. Then $Z|+\rangle = |-\rangle$: the arrow swings from $+x$ to $-x$.

In [ ]:
plus = QuantumCircuit(1)
plus.h(0)
before_after("z", init=plus)

**S and T** are smaller rotations about the same axis. Starting from $|+\rangle$, S turns the arrow a quarter-turn to $+y$, and T turns it an eighth.

In [ ]:
before_after("s", init=plus)
before_after("t", init=plus)

**Rotation gates** take an arbitrary angle. They matter a great deal later in the week: in VQE, QAOA and VQC the angle becomes a *parameter* that a classical optimiser tunes.

In [ ]:
before_after("rx", args=(np.pi/3,))

In [ ]:
# Sweep Ry(theta) and watch the arrow travel from the north pole through the x-axis to the south pole
fig, axes = plt.subplots(1, 4, figsize=(14, 3.6))
for ax, th in zip(axes, [0, np.pi/2, np.pi, 3*np.pi/2]):
    qc = QuantumCircuit(1); qc.ry(th, 0)
    f = plot_bloch_multivector(Statevector(qc), title=f"Ry({th/np.pi:.1f} pi)")
    ax.imshow(fig_to_img(f)); ax.axis("off"); plt.close(f)
plt.show()

### Gates compose by matrix multiplication

Applying H, then Z, then H is the matrix product $H\,Z\,H$. Order matters, and the product is read **right to left** because the rightmost matrix acts first on the state. A guess worth testing: conjugating Z by H should turn a "phase flip" into a "bit flip".

In [ ]:
qc = QuantumCircuit(1)
qc.h(0)
qc.z(0)
qc.h(0)
display(qc.draw("mpl"))

U = Operator(qc).data
print(np.round(U, 3))
print("H Z H equals X?", np.allclose(U, Operator(XGate()).data))

This identity says that Z in the X basis looks like X in the Z basis. We will use it in Chapter 7.

### Exercise 3: finding a gate sequence

**Goal.** Build a one-qubit circuit `ex3_circuit` that starts in $|0\rangle$ and ends in the state `"l"`, i.e. $(|0\rangle - i|1\rangle)/\sqrt2$, the $-y$ point on the sphere.

You may use `h`, `s`, `sdg` (the inverse of S), `x`, `z`, and so on. Plan it on the sphere first: which two rotations bring the arrow from the north pole to $-y$?

<details>
<summary>Hint</summary>

H takes the north pole to $+x$. Which gate rotates $+x$ about z towards $-y$? S rotates $+x$ to $+y$, so you need the opposite rotation.
</details>

In [ ]:
ex3_circuit = None     # e.g.  ex3_circuit = QuantumCircuit(1); ex3_circuit.h(0); ...

In [ ]:
check("final state is |l> (the -y point), up to global phase",
      lambda: Statevector(ex3_circuit).equiv(Statevector.from_label("l")))

<details>
<summary>Solution</summary>

```python
ex3_circuit = QuantumCircuit(1)
ex3_circuit.h(0)      # north pole -> +x
ex3_circuit.sdg(0)    # +x -> -y
```

An equivalent route is `ry(pi/2)` followed by `rz(-pi/2)`. There is more than one circuit for the same state.
</details>

**Second part.** Without running anything, predict what `qc.h(0); qc.h(0)` does. Then check with `Operator`.

In [ ]:
# your check for H followed by H

Further reading: [Construct circuits](https://quantum.cloud.ibm.com/docs/en/guides/construct-circuits) and the [gate library in the Qiskit API reference](https://quantum.cloud.ibm.com/docs/en/api/qiskit/circuit_library).

## Chapter 4. Measurement

Measurement is the interface between the quantum state and the classical world. Three rules cover almost everything we will do:

1. A measurement returns a classical bit, with probabilities given by the Born rule.
2. Afterwards the state has **collapsed** to the outcome that was observed. Measuring again gives the same result.
3. One run tells us almost nothing. We repeat the whole experiment many times; each repetition is a **shot**, and the tally of outcomes is the **counts** dictionary.

> **Physics note.** The collapse rule is von Neumann's projection postulate. In Stern-Gerlach terms: send atoms through a magnet oriented along z and keep the "up" beam. Now pass them through a magnet oriented along x and keep "up". Pass them again through z. The z information from the first stage has been erased, and the third magnet splits the beam 50/50 again. Measuring one property of a quantum system can destroy knowledge of another. We reproduce exactly this behaviour with circuits below.

### Primitives: the modern way to run circuits

Older Qiskit tutorials call `execute()`. That function no longer exists. Today you run circuits through two **primitives**:

| Primitive | You give it | You get back |
|---|---|---|
| `Sampler` | a circuit with measurements | bitstring samples and counts |
| `Estimator` | a circuit and an observable such as $Z\otimes Z$ | an expectation value $\langle O\rangle$ |

Today we use the exact reference implementations `StatevectorSampler` and `StatevectorEstimator`. Later we swap in a noisy simulator, and finally real hardware, with only small changes to the code.

Let us measure a qubit in the state $|+\rangle$.

In [ ]:
qc = QuantumCircuit(1,1)   # one qubit and one classical bit to store the result
qc.h(0)                    # put qubit 0 into the |+> state
qc.measure(0,0)            # measure qubit 0 and store the outcome in classical bit 0
display(qc.draw("mpl"))

In [ ]:
sampler = StatevectorSampler()
job = sampler.run([qc], shots=1000)    # we pass a list because the sampler can batch several circuits at once
result = job.result()[0]               # result for the first circuit in the list

counts = result.data.c.get_counts()    # "c" is the name Qiskit gave our classical register
print(counts)
plot_histogram(counts)

The attribute name `.c` is the name of the classical register. We created the register implicitly with `QuantumCircuit(1,1)`, and Qiskit calls it `c`. If instead you use `qc.measure_all()`, Qiskit creates a register called `meas` and you read `result.data.meas`. If you are ever unsure, `result.data` lists the register names.

### More shots, closer to theory

The true probability is $P(0)=1/2$, but with few shots the observed frequency fluctuates. The statistical error shrinks as $1/\sqrt{N}$.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.2))
for ax, n in zip(axes, [10, 100, 10000]):
    c = sampler.run([qc], shots=n).result()[0].data.c.get_counts()
    plot_histogram(c, ax=ax, title=f"{n} shots")
plt.tight_layout()
plt.show()

### Collapse, simulated

A `Statevector` can mimic a single measurement. It returns the outcome and the post-measurement state. Notice that the state afterwards is always a definite basis state.

In [ ]:
sv = Statevector.from_label("+")
for i in range(5):
    outcome, post = sv.measure()
    print(f"run {i}: outcome = {outcome},  state afterwards = {np.round(post.data, 3)}")

### Measuring in a different basis

Hardware measures only in the Z basis. To measure another observable we rotate the qubit first, so that the basis we care about is mapped onto Z.

| To measure in | Insert before the measurement |
|---|---|
| Z | nothing |
| X | `h` |
| Y | `sdg`, then `h` |

> **Predict.** The state $|+\rangle$ is measured in the Z basis, and then again in the X basis. Which of the two gives a 50/50 histogram, and which gives a single bar?

In [ ]:
def run(circ, shots=2000):
    return sampler.run([circ], shots=shots).result()[0].data.meas.get_counts()

# |+> measured in the Z basis: random
z_meas = QuantumCircuit(1)
z_meas.h(0)
z_meas.measure_all()

# |+> measured in the X basis: rotate X onto Z with H first
x_meas = QuantumCircuit(1)
x_meas.h(0)            # prepare |+>
x_meas.h(0)            # basis change
x_meas.measure_all()

plot_histogram([run(z_meas), run(x_meas)], legend=["Z basis", "X basis"])

The state $|+\rangle$ is an eigenstate of X, so an X measurement is certain. It is not an eigenstate of Z, so a Z measurement is random. This is the same phenomenon as in the Stern-Gerlach note above.

### Expectation values with the Estimator

Rather than counting shots ourselves, we can ask for $\langle\psi|O|\psi\rangle$ directly. For $|+\rangle$ we expect $\langle X\rangle=1$ and $\langle Y\rangle=\langle Z\rangle=0$. The Estimator needs no measurement gates in the circuit.

In [ ]:
estimator = StatevectorEstimator()

qc_plus = QuantumCircuit(1)
qc_plus.h(0)

for label in ["X", "Y", "Z"]:
    obs = SparsePauliOp(label)
    val = estimator.run([(qc_plus, obs)]).result()[0].data.evs
    print(f"<{label}> = {float(val):+.3f}")

### Exercise 4: Sampler or Estimator?

**Goal.** Prepare the state $|r\rangle = (|0\rangle + i|1\rangle)/\sqrt2$ in a circuit `ex4_circuit` (without measurements) and use the Estimator to confirm that $\langle Y\rangle = +1$ and $\langle Z\rangle = 0$.

Then answer in a comment: how would you obtain the same information using only the Sampler? (Hint: use the table of basis changes above, and note that you would need several shots, whereas the Estimator on a statevector simulator gives the exact number.)

<details>
<summary>Hint</summary>

H gives $|+\rangle$, then S rotates it a quarter-turn about z.
</details>

In [ ]:
ex4_circuit = None

In [ ]:
def _ex4():
    ev = lambda p: float(StatevectorEstimator().run([(ex4_circuit, SparsePauliOp(p))]).result()[0].data.evs)
    return np.isclose(ev("Y"), 1) and np.isclose(ev("Z"), 0)

check("<Y> = +1 and <Z> = 0", _ex4)

<details>
<summary>Solution</summary>

```python
ex4_circuit = QuantumCircuit(1)
ex4_circuit.h(0)
ex4_circuit.s(0)
```

With the Sampler alone: append `sdg` then `h`, measure, and compute $\langle Y\rangle = P(0) - P(1)$ from the counts.
</details>

Further reading: [Introduction to primitives](https://quantum.cloud.ibm.com/docs/en/guides/primitives), [Sampler quickstart](https://quantum.cloud.ibm.com/docs/en/guides/get-started-with-sampler) and [Estimator quickstart](https://quantum.cloud.ibm.com/docs/en/guides/get-started-with-estimator).

## Chapter 5. Your first circuit

IBM describes every quantum program with a four-step pattern. We will follow it all the way through the notebook.

1. **Map.** Express your problem as circuits and observables.
2. **Optimise.** Transpile the circuit for a specific target device.
3. **Execute.** Run it with a primitive.
4. **Post-process.** Turn the raw output into an answer.

### Step 1: Map

We build a three-qubit circuit.

In [ ]:
qc = QuantumCircuit(3)     # three qubits, all in |0>
qc.h(0)                    # superposition on qubit 0
qc.cx(0, 1)                # CNOT: control = qubit 0, target = qubit 1
qc.x(2)                    # flip qubit 2
qc.barrier()               # visual separator, no physical effect
qc.measure_all()

qc.draw("mpl")

In [ ]:
print("Qubits:      ", qc.num_qubits)
print("Depth:       ", qc.depth())
print("Size:        ", qc.size())
print("Gate counts: ", dict(qc.count_ops()))
print()
print(qc.draw("text"))

> **Qiskit uses little-endian bit ordering.** Qubit 0 is the **rightmost** bit of every bitstring, so strings read $q_2 q_1 q_0$. Keep this in mind whenever you interpret a histogram.

> **Predict.** Qubit 2 is always 1. Qubits 0 and 1 are correlated. Which bitstrings do you expect, and with what frequencies?

### Steps 2 and 3: Optimise and Execute

The circuit has to be rewritten in terms of the gates a backend actually supports. This process is called transpilation, and its output is sometimes called an **ISA circuit** (Instruction Set Architecture). Here the backend is Aer's ideal simulator.

In [ ]:
from qiskit_aer import AerSimulator
from qiskit_aer.primitives import SamplerV2 as AerSampler
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

backend = AerSimulator()
pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_qc = pm.run(qc)

aer_sampler = AerSampler()
result = aer_sampler.run([isa_qc], shots=2000).result()[0]
counts = result.data.meas.get_counts()
print(counts)

### Step 4: Post-process

In [ ]:
plot_histogram(counts, title="Our first circuit")

Qubit 2 is 1 in every outcome, and qubits 0 and 1 always agree. Reading $q_2q_1q_0$ gives `100` and `111`, each about half the time. Did that match your prediction?

### Parametrised circuits

A circuit may contain **named placeholders** that are filled in at run time. This is the mechanism behind variational algorithms. Below, the angle of an $R_y$ gate is swept over 25 values in a single call. The theory is $P(1) = \sin^2(\theta/2)$.

In [ ]:
from qiskit.circuit import Parameter

theta = Parameter("θ")
pqc = QuantumCircuit(1)
pqc.ry(theta, 0)
pqc.measure_all()
display(pqc.draw("mpl"))

thetas = np.linspace(0, 2*np.pi, 25)
sampler = StatevectorSampler()
data = sampler.run([(pqc, thetas.reshape(-1, 1))], shots=2000).result()[0].data.meas

p1 = [data[i].get_counts().get("1", 0) / 2000 for i in range(len(thetas))]

fine = np.linspace(0, 2*np.pi, 200)
plt.plot(thetas, p1, "o", label="simulated shots")
plt.plot(fine, np.sin(fine/2)**2, "-", label=r"theory $\sin^2(\theta/2)$")
plt.xlabel("theta"); plt.ylabel("P(1)")
plt.legend(); plt.grid(alpha=.3); plt.show()

### Exercise 5: your own circuit

**Goal.** Build a two-qubit circuit `ex5_circuit` that, when measured, returns `"11"` with certainty. Then build a second circuit `ex5b_circuit` that returns `"01"` with certainty, and think carefully about which qubit you flipped.

<details>
<summary>Hint</summary>

Start from $|00\rangle$ and use X gates. Remember the bitstring is $q_1 q_0$. The string `"01"` means qubit 0 is 1 and qubit 1 is 0.
</details>

In [ ]:
ex5_circuit  = None     # should give "11"
ex5b_circuit = None     # should give "01"

In [ ]:
check('circuit 1 gives "11" with probability 1',
      lambda: Statevector(ex5_circuit).probabilities_dict() == {"11": 1.0})
check('circuit 2 gives "01" with probability 1',
      lambda: Statevector(ex5b_circuit).probabilities_dict() == {"01": 1.0})

<details>
<summary>Solution</summary>

```python
ex5_circuit = QuantumCircuit(2)
ex5_circuit.x(0)
ex5_circuit.x(1)

ex5b_circuit = QuantumCircuit(2)
ex5b_circuit.x(0)          # qubit 0 is the RIGHTMOST bit, so this gives "01"
```
</details>

Further reading: [Qiskit patterns and the transpiler](https://quantum.cloud.ibm.com/docs/en/guides/transpile) and the [primitives API reference](https://quantum.cloud.ibm.com/docs/en/api/qiskit/primitives).

## Chapter 6. Entanglement

A two-qubit state is **entangled** when it cannot be written as a product of one-qubit states. The simplest example is the Bell state

$$|\Phi^+\rangle = \frac{|00\rangle + |11\rangle}{\sqrt2}.$$

You cannot find any $(\alpha,\beta)$ and $(\gamma,\delta)$ such that $(\alpha|0\rangle+\beta|1\rangle)\otimes(\gamma|0\rangle+\delta|1\rangle)$ equals it. Try: it would need $\alpha\delta=0$ yet $\alpha\gamma\neq0$ and $\beta\delta\neq0$.

We prepare it with a Hadamard and a CNOT. The CNOT flips the target qubit when the control is $|1\rangle$:

$$\text{CNOT} = \begin{pmatrix}1&0&0&0\\0&1&0&0\\0&0&0&1\\0&0&1&0\end{pmatrix}.$$

Following the state through the circuit:

$$|00\rangle \xrightarrow{H\otimes I} \tfrac{1}{\sqrt2}(|00\rangle + |10\rangle) \xrightarrow{\text{CNOT}} \tfrac{1}{\sqrt2}(|00\rangle + |11\rangle).$$

> **Physics note.** Einstein, Podolsky and Rosen (1935) argued that such correlations meant quantum mechanics was incomplete. In 1964 John Bell showed that the question could be settled by experiment: any theory with pre-existing local "hidden variables" obeys an inequality that quantum mechanics violates. Clauser, Horne, Shimony and Holt (1969) put it in an experimentally testable form, and Alain Aspect's experiments in the early 1980s gave convincing violations. Aspect, Clauser and Zeilinger shared the 2022 Nobel Prize in Physics for this line of work. We will run the same test below.

In [ ]:
bell = QuantumCircuit(2)
bell.h(0)
bell.cx(0, 1)
display(bell.draw("mpl"))

sv = Statevector(bell)
display(sv.draw("latex"))
print(sv.probabilities_dict())

### Each qubit alone looks completely random

Look at the Bloch spheres of the two qubits separately. Both arrows have **shrunk to the centre** of the sphere. A pure state always sits on the surface, so this tells us that a single qubit of a Bell pair is not in a pure state. All the information is in the correlation, none in the individual parts.

In [ ]:
plot_bloch_multivector(sv, title="Bell state: each qubit on its own")

In [ ]:
# For contrast, a product state |+>|+> where each arrow reaches the surface
prod = QuantumCircuit(2)
prod.h(0)
prod.h(1)
plot_bloch_multivector(Statevector(prod), title="Product state |+>|+>")

### Measuring the pair

Each outcome individually is random, but the two always agree. We see `00` and `11` and never `01` or `10`.

In [ ]:
bell_m = bell.copy()
bell_m.measure_all()
counts = StatevectorSampler().run([bell_m], shots=2000).result()[0].data.meas.get_counts()
plot_histogram(counts, title="Bell state measured in the Z basis")

### A number for entanglement

The **entanglement entropy** of one qubit is the von Neumann entropy $S=-\mathrm{Tr}(\rho\log_2\rho)$ of its reduced density matrix. It is 0 for a product state and 1 for a maximally entangled pair.

In [ ]:
from qiskit.quantum_info import partial_trace, entropy

for name, c in [("product |+>|+>", prod), ("Bell pair", bell)]:
    rho_q0 = partial_trace(Statevector(c), [1])      # trace out qubit 1
    print(f"{name:15s}  entropy of qubit 0 = {entropy(rho_q0, base=2):.3f}")

### Many qubits: the GHZ state

$$|\mathrm{GHZ}_n\rangle = \frac{|0\ldots0\rangle + |1\ldots1\rangle}{\sqrt2}$$

is built by one Hadamard followed by a chain of CNOTs.

In [ ]:
n = 5
ghz = QuantumCircuit(n)
ghz.h(0)
for i in range(n - 1):
    ghz.cx(i, i + 1)
display(ghz.draw("mpl"))

ghz_m = ghz.copy()
ghz_m.measure_all()
plot_histogram(StatevectorSampler().run([ghz_m], shots=2000).result()[0].data.meas.get_counts())

### The CHSH test

Choose two measurement settings for Alice ($A_0, A_1$) and two for Bob ($B_0, B_1$), and form

$$S = \langle A_0B_0\rangle + \langle A_0B_1\rangle + \langle A_1B_0\rangle - \langle A_1B_1\rangle .$$

Any local hidden-variable theory obeys $|S|\le 2$. Quantum mechanics allows up to $2\sqrt2 \approx 2.83$ (Tsirelson's bound).

With $A_0=Z,\ A_1=X,\ B_0=(Z+X)/\sqrt2,\ B_1=(Z-X)/\sqrt2$, the algebra collapses to

$$S = \sqrt2\,\big(\langle Z\otimes Z\rangle + \langle X\otimes X\rangle\big).$$

The Estimator evaluates this directly.

In [ ]:
chsh = SparsePauliOp.from_list([("ZZ", np.sqrt(2)), ("XX", np.sqrt(2))])

S = float(StatevectorEstimator().run([(bell, chsh)]).result()[0].data.evs)
print(f"CHSH value S      = {S:.4f}")
print(f"classical bound   = 2")
print(f"quantum maximum   = {2*np.sqrt(2):.4f}")
print("Bell inequality violated." if S > 2 else "No violation.")

> The value 2.83 is not a numerical artefact. It is the quantum prediction, and it is what real experiments (including ones on IBM hardware) approach, reduced by noise.

### Exercise 6: the four Bell states

The four maximally entangled two-qubit states are

$$|\Phi^\pm\rangle = \frac{|00\rangle \pm |11\rangle}{\sqrt2}, \qquad |\Psi^\pm\rangle = \frac{|01\rangle \pm |10\rangle}{\sqrt2}.$$

**Goal.** Build circuits `bell_phi_minus`, `bell_psi_plus` and `bell_psi_minus`. Each should be the standard Bell circuit with extra single-qubit gates added. Work out on paper first which gate changes the sign and which swaps $00,11$ into $01,10$.

<details>
<summary>Hint</summary>

Putting X on a qubit before the entangling step changes which pair of bitstrings appears. A Z (or X, then Z) afterwards changes the relative sign. Remember qubit 0 is the rightmost bit when you compare with the target vectors.
</details>

In [ ]:
bell_phi_minus = None
bell_psi_plus  = None
bell_psi_minus = None

In [ ]:
s = 1/np.sqrt(2)
targets = {
    "bell_phi_minus": ([ s, 0, 0, -s], lambda: bell_phi_minus),
    "bell_psi_plus":  ([ 0, s, s,  0], lambda: bell_psi_plus),
    "bell_psi_minus": ([ 0, s, -s, 0], lambda: bell_psi_minus),
}
for name, (vec, getter) in targets.items():
    check(name, lambda g=getter, v=vec: Statevector(g()).equiv(Statevector(v)))

<details>
<summary>Solution</summary>

```python
bell_phi_minus = QuantumCircuit(2)
bell_phi_minus.h(0); bell_phi_minus.cx(0, 1); bell_phi_minus.z(0)

bell_psi_plus = QuantumCircuit(2)
bell_psi_plus.h(0); bell_psi_plus.cx(0, 1); bell_psi_plus.x(0)

bell_psi_minus = QuantumCircuit(2)
bell_psi_minus.h(0); bell_psi_minus.cx(0, 1); bell_psi_minus.x(0); bell_psi_minus.z(0)
```

Up to a global phase these are all equivalent to the answers above. Notice that the four Bell states differ only by **local** operations on one qubit. That is the idea behind superdense coding.
</details>

**Stretch.** Compute the CHSH value of each of the four states with the same operator `chsh`. Which ones violate the bound with this particular choice of settings, and why do you think some do not?

Further reading: [Qiskit quantum_info module](https://quantum.cloud.ibm.com/docs/en/api/qiskit/quantum_info), and the course [Basics of quantum information](https://quantum.cloud.ibm.com/learning/en/courses/basics-of-quantum-information) for a proof of Bell's inequality and the CHSH game.

## Chapter 7. Interference

If a quantum computer were only a source of random numbers it would not be useful. The resource that makes algorithms work is **interference**. Because amplitudes can be negative or complex, two routes that lead to the same outcome can add up (constructive interference) or cancel (destructive interference).

> **Physics note.** This is Young's double slit (1801) and the Mach-Zehnder interferometer in the language of circuits. Feynman's formulation says the amplitude for an event is the sum of amplitudes over all ways it can happen. Probabilities are squares of these sums, so the cross terms, the interference terms, carry real consequences.

Take a qubit in $|0\rangle$ and apply a Hadamard twice:

$$H H|0\rangle = H\,\frac{|0\rangle+|1\rangle}{\sqrt2} = \frac{(|0\rangle+|1\rangle) + (|0\rangle-|1\rangle)}{2} = |0\rangle.$$

The $|1\rangle$ outcome has two routes. One contributes $+\tfrac12$ and the other $-\tfrac12$, so they cancel. A classical coin flipped twice stays random; a qubit passed through $H$ twice returns to where it began.

> **Predict.** After H, H and a measurement, how many shots out of 1000 return 1?

In [ ]:
qc = QuantumCircuit(1)
qc.h(0)
qc.h(0)
qc.measure_all()
display(qc.draw("mpl"))

counts = StatevectorSampler().run([qc], shots=1000).result()[0].data.meas.get_counts()
print(counts)
plot_histogram(counts, title="H, then H: always 0")

### Tuning the interference with a phase

Put a controllable phase $\varphi$ between the two Hadamards.

$$|0\rangle \xrightarrow{H} \tfrac{1}{\sqrt2}(|0\rangle+|1\rangle) \xrightarrow{P(\varphi)} \tfrac{1}{\sqrt2}(|0\rangle+e^{i\varphi}|1\rangle) \xrightarrow{H}\ \ P(0)=\cos^2\tfrac{\varphi}{2}.$$

At $\varphi=0$ the two routes to $|0\rangle$ add. At $\varphi=\pi$ they cancel and every shot returns 1. The phase on $|1\rangle$ was invisible to a direct Z measurement, but the second Hadamard converts it into a probability. This answers the question we asked in Chapter 1 about `"+"` and `"-"`.

Run the interferometer for many phases.

In [ ]:
phi = Parameter("φ")
mz = QuantumCircuit(1)
mz.h(0)
mz.p(phi, 0)
mz.h(0)
mz.measure_all()
display(mz.draw("mpl"))

phis = np.linspace(0, 4*np.pi, 41)
res = StatevectorSampler().run([(mz, phis.reshape(-1, 1))], shots=4000).result()[0].data.meas
p0 = [res[i].get_counts().get("0", 0) / 4000 for i in range(len(phis))]

fine = np.linspace(0, 4*np.pi, 300)
plt.figure(figsize=(7, 3.5))
plt.plot(phis, p0, "o", label="simulated shots")
plt.plot(fine, np.cos(fine/2)**2, label=r"theory $\cos^2(\varphi/2)$")
plt.xlabel("phase phi"); plt.ylabel("P(0)")
plt.title("Interference fringes")
plt.legend(); plt.grid(alpha=.3); plt.show()

### Seeing the phase on the sphere

Drag the phase slider. On the Bloch sphere the arrow slides around the equator, which changes nothing a Z measurement can see. The probability printed underneath is what the *final* Hadamard would reveal.

In [ ]:
@interact(phase=FloatSlider(min=0, max=2*np.pi, step=0.1, value=0.0, description="phase"))
def fringe(phase):
    q = QuantumCircuit(1)
    q.h(0)
    q.p(phase, 0)
    fig = plot_bloch_multivector(Statevector(q), title=f"phase = {phase/np.pi:.2f} pi")
    display(fig)
    plt.close(fig)
    q.h(0)
    print(f"after the second H:  P(0) = {Statevector(q).probabilities()[0]:.3f}")

Algorithms such as Deutsch-Jozsa, the quantum Fourier transform, phase estimation and Grover search are all built on this idea. They arrange phases so that wrong answers cancel and the right answer reinforces.

### Exercise 7: engineering an interference pattern

**Part A.** Predict, then verify with a circuit `ex7a`, the measurement outcome of $H\,Z\,H$ acting on $|0\rangle$. (Use the identity from Chapter 3.)

**Part B.** Using the interferometer above, find a phase `ex7_phi` such that $P(0) = 0.25$.

<details>
<summary>Hint for Part B</summary>

$P(0)=\cos^2(\varphi/2)$. Set it equal to 0.25 and solve, as you did in Exercise 2.
</details>

In [ ]:
ex7a = None        # a QuantumCircuit with H, Z, H acting on |0>
ex7_phi = None     # a number in radians

In [ ]:
check("H Z H on |0> gives |1> with certainty",
      lambda: np.allclose(Statevector(ex7a).probabilities(), [0, 1]))

def _ex7b():
    q = QuantumCircuit(1); q.h(0); q.p(ex7_phi, 0); q.h(0)
    return np.isclose(Statevector(q).probabilities()[0], 0.25)

check("interferometer with your phase has P(0) = 0.25", _ex7b)

<details>
<summary>Solution</summary>

```python
ex7a = QuantumCircuit(1)
ex7a.h(0); ex7a.z(0); ex7a.h(0)      # equals X, so |0> -> |1>

ex7_phi = 2*np.pi/3                  # cos^2(pi/3) = 0.25
```

Two solutions exist in $[0,2\pi)$: $2\pi/3$ and $4\pi/3$. Why? Look at the fringe plot.
</details>

Further reading: the course [Fundamentals of quantum algorithms](https://quantum.cloud.ibm.com/learning/en/courses/fundamentals-of-quantum-algorithms) starts exactly from this interference mechanism.

## Chapter 8. Noise and real devices

Everything so far was ideal. Real devices are in the **NISQ** regime (noisy, intermediate-scale quantum), and several effects spoil the clean pictures above.

| Source | What happens |
|---|---|
| Gate errors | Pulses are not exactly the intended rotation. Typical error rates run from about 0.01% to 1%. |
| Relaxation ($T_1$) | An excited qubit decays $|1\rangle\to|0\rangle$ by losing energy. |
| Dephasing ($T_2$) | The relative phase between $|0\rangle$ and $|1\rangle$ drifts and interference washes out. |
| Readout error | A qubit in $|0\rangle$ is reported as 1, or vice versa. |
| Crosstalk | Operations on one qubit disturb its neighbours. |

> **Physics note.** The names $T_1$ and $T_2$ come directly from NMR, where Bloch introduced them. On the Bloch sphere, relaxation drags the arrow towards the north pole and dephasing shrinks it towards the vertical axis. More generally, noise is the result of the qubit being entangled with an environment that we do not observe, which is the decoherence picture developed by Zurek and others. It is the same entanglement as in Chapter 6, now working against us. This is also why Feynman's original 1981 proposal asked for a *quantum* simulator: a classical machine cannot efficiently track these correlations.

### A custom noise model

In [ ]:
from qiskit_aer.noise import NoiseModel, depolarizing_error, ReadoutError

noise = NoiseModel()

# One-qubit gates: 0.1% depolarising error. Two-qubit gates: 2%.
noise.add_all_qubit_quantum_error(depolarizing_error(0.001, 1), ["h", "x", "sx", "rz"])
noise.add_all_qubit_quantum_error(depolarizing_error(0.02, 2),  ["cx"])

# Readout: P(read 1 | true 0) = 2%, P(read 0 | true 1) = 4%
noise.add_all_qubit_readout_error(ReadoutError([[0.98, 0.02], [0.04, 0.96]]))

print(noise)

In [ ]:
ideal_backend = AerSimulator()
noisy_backend = AerSimulator(noise_model=noise)

bell_m = QuantumCircuit(2)
bell_m.h(0); bell_m.cx(0, 1); bell_m.measure_all()

pm_ideal = generate_preset_pass_manager(backend=ideal_backend)
pm_noisy = generate_preset_pass_manager(optimization_level=1,
                                        basis_gates=["h", "x", "sx", "rz", "cx"])

c_ideal = ideal_backend.run(pm_ideal.run(bell_m), shots=4000).result().get_counts()
c_noisy = noisy_backend.run(pm_noisy.run(bell_m), shots=4000).result().get_counts()

plot_histogram([c_ideal, c_noisy], legend=["ideal", "noisy"], title="Bell state, ideal vs noisy")

The outcomes `01` and `10` now appear, even though they should be impossible. Every one of them is an error.

### Noise grows with circuit size

An $n$-qubit GHZ circuit needs $n-1$ CNOTs. With a 2% error per CNOT, how does the fraction of correct outcomes (`00...0` or `11...1`) fall as $n$ grows?

> **Predict.** If each CNOT succeeds with probability 0.98 and errors compound independently, roughly what success rate would you expect for $n=10$? (Hint: $0.98^9$.)

In [ ]:
def ghz_circuit(n):
    qc = QuantumCircuit(n)
    qc.h(0)
    for i in range(n - 1):
        qc.cx(i, i + 1)
    qc.measure_all()
    return qc

ns = list(range(2, 11))
good = []
for n in ns:
    c = noisy_backend.run(pm_noisy.run(ghz_circuit(n)), shots=4000).result().get_counts()
    good.append((c.get("0"*n, 0) + c.get("1"*n, 0)) / 4000)

plt.plot(ns, good, "o-", label="noisy simulation")
plt.axhline(1, ls="--", c="gray", label="ideal")
plt.xlabel("number of qubits n"); plt.ylabel("fraction of correct outcomes")
plt.title("GHZ success rate against size")
plt.legend(); plt.grid(alpha=.3); plt.show()

### Fake backends: rehearsing on a specific device

`qiskit-ibm-runtime` includes **snapshots of real IBM devices**: their connectivity, native gates and calibrated error rates. You can use them to see how your circuit will behave on that machine, without an account or a queue.

In [ ]:
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke

fake = FakeSherbrooke()
print("Name:         ", fake.name)
print("Qubits:       ", fake.num_qubits)
print("Native gates: ", fake.operation_names)

### Transpilation

Real devices implement only a small set of **native gates** and allow two-qubit gates only between **physically connected** qubits. The transpiler rewrites your circuit to respect both. Below, qubit 0 must interact with all three others, which is not possible directly on a sparse chip. Compare the circuit before and after.

In [ ]:
ghz4 = QuantumCircuit(4)
ghz4.h(0); ghz4.cx(0, 1); ghz4.cx(0, 2); ghz4.cx(0, 3)
ghz4.measure_all()

print("Logical circuit:")
display(ghz4.draw("mpl", fold=-1))

pm = generate_preset_pass_manager(backend=fake, optimization_level=3, seed_transpiler=42)
isa = pm.run(ghz4)

print("After transpilation for the device:")
display(isa.draw("mpl", idle_wires=False, fold=-1))
print("Depth:", ghz4.depth(), "->", isa.depth())
print("Gates after:", dict(isa.count_ops()))

In [ ]:
noisy_sim = AerSimulator.from_backend(fake)      # Aer loaded with the device's noise data
counts = noisy_sim.run(isa, shots=4000).result().get_counts()
plot_histogram(counts, title="GHZ-4 on a simulated IBM Sherbrooke", figsize=(10, 4), number_to_keep=8)

The transpiler has several optimisation levels.

| Level | Behaviour |
|---|---|
| 0 | Map to the device only. |
| 1 | Light optimisation. |
| 2 | Better qubit placement and gate cancellation. |
| 3 | Heaviest optimisation, slowest to compile. |

Higher is not always better for every circuit, so it is worth comparing, as below.

In [ ]:
for lvl in range(4):
    t = generate_preset_pass_manager(backend=fake, optimization_level=lvl, seed_transpiler=7).run(ghz4)
    ops = t.count_ops()
    print(f"level {lvl}:  depth = {t.depth():3d}   two-qubit gates = {ops.get('ecr', 0) + ops.get('cz', 0)}")

**Error suppression and mitigation.** Techniques such as dynamical decoupling, gate twirling and zero-noise extrapolation reduce the effect of noise without full error correction. In `qiskit-ibm-runtime` they are switched on through primitive options, and we will use two of them in Chapter 9.

### Exercise 8: which error matters most?

In the model above the CNOT error is 2% and the readout error is a few percent per qubit.

**Goal.** Reduce the CNOT error to 0.2% (ten times better) and compute the GHZ success rate for $n=10$ in two cases:

- `ex8_with_readout`: keeping the readout errors,
- `ex8_no_readout`: with the readout errors removed.

First predict the second number from $0.998^9$. Then explain in a comment why the first number is so much lower. Which error source would you work on first?

<details>
<summary>Hint</summary>

Copy the `NoiseModel` cell and change only the number inside `depolarizing_error(0.02, 2)`. For the second case leave out the `add_all_qubit_readout_error` line. Use `ghz_circuit(10)` and the transpiler `pm_noisy` defined above.
</details>

In [ ]:
ex8_with_readout = None
ex8_no_readout   = None

In [ ]:
check("without readout error the success rate is above 0.95", lambda: ex8_no_readout > 0.95)
check("readout error lowers the success rate by at least 0.1", lambda: ex8_no_readout - ex8_with_readout > 0.1)

<details>
<summary>Solution</summary>

```python
def ghz_success(cx_error, readout=True, n=10, shots=4000):
    nm = NoiseModel()
    nm.add_all_qubit_quantum_error(depolarizing_error(0.001, 1), ["h", "x", "sx", "rz"])
    nm.add_all_qubit_quantum_error(depolarizing_error(cx_error, 2), ["cx"])
    if readout:
        nm.add_all_qubit_readout_error(ReadoutError([[0.98, 0.02], [0.04, 0.96]]))
    c = AerSimulator(noise_model=nm).run(pm_noisy.run(ghz_circuit(n)), shots=shots).result().get_counts()
    return (c.get("0"*n, 0) + c.get("1"*n, 0)) / shots

ex8_with_readout = ghz_success(0.002, readout=True)     # about 0.73
ex8_no_readout   = ghz_success(0.002, readout=False)    # about 0.99
```

With the original 2% CNOT error the same experiment gives about 0.66 with readout error and 0.87 without. Making the CNOTs ten times better lifts the no-readout number to about 0.99, but with readout error included we only reach about 0.73, because each of the ten qubits is misread a few percent of the time. At that point readout error dominates, which is why **readout error mitigation** is one of the first techniques applied on real hardware.
</details>

Further reading: [Build noise models](https://quantum.cloud.ibm.com/docs/en/guides/build-noise-models), [Introduction to transpilation](https://quantum.cloud.ibm.com/docs/en/guides/transpile), [Local testing mode](https://quantum.cloud.ibm.com/docs/en/guides/local-testing-mode) and [Error mitigation and suppression](https://quantum.cloud.ibm.com/docs/en/guides/error-mitigation-and-suppression-techniques).

## Chapter 9. Running on IBM Quantum hardware

Everything in this chapter needs an IBM Quantum Platform account, so the code is guarded by a switch that is **off by default**. The rest of the notebook runs without an account.

### Setting up

1. Create an account at [quantum.cloud.ibm.com](https://quantum.cloud.ibm.com).
2. From the dashboard, copy your **API key**. If the page shows an instance name or CRN, note that too.
3. Set `RUN_ON_HARDWARE = True` below and paste the key when prompted. Treat the key like a password and never paste it into a shared notebook.

The free Open Plan provides a monthly allowance of quantum compute time. See [Set up your account](https://quantum.cloud.ibm.com/docs/en/guides/cloud-setup) and [Initialize your account](https://quantum.cloud.ibm.com/docs/en/guides/initialize-account) for the current details.

In [ ]:
RUN_ON_HARDWARE = False        # change to True once you have your API key

In [ ]:
from qiskit_ibm_runtime import QiskitRuntimeService

if RUN_ON_HARDWARE:
    from getpass import getpass
    token = getpass("Paste your IBM Quantum API key: ")

    QiskitRuntimeService.save_account(
        channel="ibm_quantum_platform",
        token=token,
        overwrite=True,
        set_as_default=True,
    )
    service = QiskitRuntimeService()
    print("Connected. Available backends:")
    for b in service.backends():
        print(f"  {b.name:20s} qubits = {b.num_qubits}")

In [ ]:
if RUN_ON_HARDWARE:
    backend = service.least_busy(operational=True, simulator=False, min_num_qubits=5)
    print("Using:", backend.name, "| pending jobs:", backend.status().pending_jobs)

### The same four steps

We run the Bell circuit from Chapter 6. Compare it with Chapter 5: Map, Optimise, Execute, Post-process, and only one line changes. In `SamplerV2(mode=backend)` the `backend` is now a real device.

In [ ]:
if RUN_ON_HARDWARE:
    from qiskit_ibm_runtime import SamplerV2 as RuntimeSampler

    # 1. Map
    bell_hw = QuantumCircuit(2)
    bell_hw.h(0); bell_hw.cx(0, 1); bell_hw.measure_all()

    # 2. Optimise for this specific device
    pm_hw = generate_preset_pass_manager(backend=backend, optimization_level=3)
    isa_hw = pm_hw.run(bell_hw)

    # 3. Execute
    sampler_hw = RuntimeSampler(mode=backend)
    sampler_hw.options.dynamical_decoupling.enable = True     # error suppression
    sampler_hw.options.twirling.enable_gates = True           # gate twirling

    job = sampler_hw.run([isa_hw], shots=2000)
    print("Job ID:", job.job_id(), "| status:", job.status())

In [ ]:
if RUN_ON_HARDWARE:
    # 4. Post-process. This waits until the job has left the queue and finished.
    hw_counts = job.result()[0].data.meas.get_counts()
    plot_histogram(hw_counts, title=f"Bell state on {backend.name}")

> **Predict.** Compare this histogram with our noisy simulation. Which of `01` and `10` do you expect to be bigger, and by how much? Real devices are not symmetric: the relaxation $|1\rangle\to|0\rangle$ makes some errors more likely than others.

### Expectation values on hardware

For observables, the same pattern uses the Estimator. A small but important detail is that the observable must be mapped onto the physical qubits that the transpiler chose.

```python
from qiskit_ibm_runtime import EstimatorV2 as RuntimeEstimator

estimator = RuntimeEstimator(mode=backend)
estimator.options.resilience_level = 2                  # include error mitigation
isa_obs = obs.apply_layout(isa_circuit.layout)          # map the observable to physical qubits
job = estimator.run([(isa_circuit, isa_obs)])
```

### Execution modes

| Mode | Use when |
|---|---|
| `mode=backend` (job mode) | Single independent jobs. Simplest, available on the Open Plan. |
| `Session` | Many dependent jobs in a row, as in an iterative optimiser. Not on the Open Plan. |
| `Batch` | Many independent jobs submitted together. |

### Simulator or hardware?

| | Simulator | Real device |
|---|---|---|
| Speed | Immediate | Queue plus run time |
| Noise | None, or modelled | Real |
| Size limit | About 30 qubits for exact statevector simulation | 100+ qubits |
| Best for | Learning, debugging, small experiments | Benchmarks and anything beyond classical simulation |

Further reading: [Execution modes](https://quantum.cloud.ibm.com/docs/en/guides/execution-modes), [Sampler quickstart](https://quantum.cloud.ibm.com/docs/en/guides/get-started-with-sampler), [Estimator quickstart](https://quantum.cloud.ibm.com/docs/en/guides/get-started-with-estimator).

## Chapter 10. A worked problem: adding two bits

We close with a small problem solved from scratch, one decision at a time. The method is more important than the answer.

**The problem.** Given two input bits $a$ and $b$, produce their sum as a two-bit number: a **sum** bit and a **carry** bit. This is called a half adder.

You already know every tool needed. Before each code cell there is a question. Think first, then open the answer.

### Step 1. State the target precisely

What should the circuit output for each of the four possible inputs? Write the truth table before touching a gate.

In [ ]:
print(" a b | carry sum")
print("-----+-----------")
for a in (0, 1):
    for b in (0, 1):
        total = a + b
        print(f" {a} {b} |   {total // 2}     {total % 2}")

### Step 2. Look for a pattern in each output column

Look at the `sum` column and the `carry` column on their own. Each is a very familiar logical operation of $a$ and $b$.

<details>
<summary>Answer</summary>

`sum` is $a \oplus b$ (XOR: 1 when the inputs differ). `carry` is $a \wedge b$ (AND: 1 only when both are 1).
</details>

### Step 3. Which gate computes XOR?

Quantum gates must be reversible. We cannot simply "overwrite" an output, so we compute into a fresh qubit that starts at 0. Consider the CNOT gate. It flips its target when the control is 1, so the target becomes `target XOR control`. What happens if the target starts as 0 and we apply a CNOT from $a$, then another from $b$?

<details>
<summary>Answer</summary>

After the first CNOT the target holds $0\oplus a = a$. After the second it holds $a\oplus b$. Two CNOTs into the same fresh qubit compute the sum bit.
</details>

Let us test that claim on all four inputs before building anything else.

In [ ]:
print(" a b | sum")
for a in (0, 1):
    for b in (0, 1):
        t = QuantumCircuit(3)           # qubit 0 = a, qubit 1 = b, qubit 2 = sum (starts at 0)
        if a: t.x(0)
        if b: t.x(1)
        t.cx(0, 2)
        t.cx(1, 2)
        bits = list(Statevector(t).probabilities_dict().keys())[0]     # e.g. "101" = q2 q1 q0
        print(f" {a} {b} |  {bits[0]}")

### Step 4. Which gate computes AND?

We need a gate that flips a fresh qubit **only if both** $a$ and $b$ are 1. A gate with two controls does exactly this. It is called the **Toffoli** gate, or CCX, and in Qiskit it is `qc.ccx(control1, control2, target)`.

Why can CNOT alone not compute AND? Think about whether you could write AND as a combination of XORs. The Toffoli gate is the smallest ingredient that makes classical reversible logic **universal**, and every classical computation can be embedded in a quantum circuit this way.

> **Physics note.** Toffoli introduced this gate in 1980 while studying reversible computation. The motivation was thermodynamic. Landauer showed that *erasing* a bit costs at least $k_B T\ln 2$ of heat, but a reversible step erases nothing. Quantum gates are unitary, hence reversible, so this line of work is the direct ancestor of quantum circuits.

### Step 5. Assemble the circuit

We need four qubits: $a$, $b$, `sum` and `carry`. We keep $a$ and $b$ unchanged, since a reversible circuit may not throw information away. We name the registers so the drawing is readable.

In [ ]:
from qiskit import QuantumRegister, ClassicalRegister

def half_adder(a_val, b_val, superposed=False):
    a     = QuantumRegister(1, "a")
    b     = QuantumRegister(1, "b")
    s     = QuantumRegister(1, "sum")
    carry = QuantumRegister(1, "carry")
    out   = ClassicalRegister(2, "out")           # out[0] = sum, out[1] = carry
    qc = QuantumCircuit(a, b, s, carry, out)

    # prepare the inputs
    if superposed:
        qc.h(a); qc.h(b)
    else:
        if a_val: qc.x(a)
        if b_val: qc.x(b)
    qc.barrier()

    # the adder itself
    qc.cx(a, s)               # sum  <- a
    qc.cx(b, s)               # sum  <- a XOR b
    qc.ccx(a, b, carry)       # carry <- a AND b
    qc.barrier()

    qc.measure(s, out[0])
    qc.measure(carry, out[1])
    return qc

half_adder(1, 1).draw("mpl")

### Step 6. Test every input

A circuit is not solved until it is tested. We run all four inputs and compare with the truth table from Step 1. The outcome string is `carry sum` because `out[1]` is on the left.

In [ ]:
sampler = StatevectorSampler()
all_correct = True

print(" a b | measured (carry sum) | expected")
for a in (0, 1):
    for b in (0, 1):
        counts = sampler.run([half_adder(a, b)], shots=100).result()[0].data.out.get_counts()
        measured = max(counts, key=counts.get)
        expected = f"{(a + b) // 2}{(a + b) % 2}"
        ok = (measured == expected) and len(counts) == 1
        all_correct &= ok
        print(f" {a} {b} |         {measured}           |   {expected}   {'ok' if ok else 'WRONG'}")

print("\nAll four rows correct:", all_correct)

### Step 7. What changes in superposition?

So far we have only fed in classical bits, and the quantum circuit behaved like a classical one. The new ingredient is that the inputs may be in superposition. Put a Hadamard on both $a$ and $b$, and the state before the adder is an equal superposition of all four input pairs:

$$\tfrac12\big(|00\rangle+|01\rangle+|10\rangle+|11\rangle\big)_{ab}\ |0\rangle_{\text{sum}}|0\rangle_{\text{carry}}.$$

Because the circuit is linear, it acts on every term at once. The result is an entangled state containing all four rows of the truth table:

$$\tfrac12\sum_{a,b}|a\rangle|b\rangle|a\oplus b\rangle|a\wedge b\rangle .$$

> **Predict.** If we now measure all four qubits, will we see one row of the truth table, or all four rows at once? What will the histogram look like, and what must be true of every bitstring in it?

In [ ]:
qc = half_adder(0, 0, superposed=True)
qc.remove_final_measurements(inplace=True)      # drop the two partial measurements
qc.measure_all()                                # measure all four qubits instead

counts = sampler.run([qc], shots=4000).result()[0].data.meas.get_counts()
plot_histogram(counts, title="Half adder on a superposition of inputs")

print("bitstring  =  carry sum b a   ->  is it a valid row?")
for bits in sorted(counts):
    carry, s, b, a = (int(x) for x in bits)
    valid = (s == (a ^ b)) and (carry == (a & b))
    print(f"   {bits}        {'valid' if valid else 'INVALID'}")

**What we learned.**

- The measurement returns **one** row, chosen at random, not all four. A single run does not hand us the whole truth table. This is the central fact about quantum parallelism: the circuit processes all inputs, but we can only read out one result.
- Every outcome is nevertheless a **valid** row. The entanglement between inputs and outputs preserves the logic.
- To get an actual speed-up one has to use interference (Chapter 7) so that a *global* property of the function survives the measurement. That is exactly what Deutsch-Jozsa, our first real algorithm of the week, does.

### Exercise 9: extend the problem

**Goal.** A *full adder* also adds a carry-in bit $c$. The sum is $a\oplus b\oplus c$ and the carry-out is 1 whenever at least two of $a,b,c$ are 1. Build a function `full_adder(a_val, b_val, c_val)` with qubits $a, b, c$, `sum`, `carry`. Think in the same sequence of steps: truth table, the pattern in each column, which gate for each.

<details>
<summary>Hint</summary>

The sum needs three CNOTs into the same fresh qubit. The carry is the majority of $a,b,c$, which equals $ab\oplus bc\oplus ac$. Each product is a Toffoli, and XORs accumulate into the same fresh qubit as before.
</details>

In [ ]:
def full_adder(a_val, b_val, c_val):
    # build and return a QuantumCircuit that measures (carry, sum) into a 2-bit register
    return None

In [ ]:
def _fa():
    from itertools import product
    for a, b, c in product((0, 1), repeat=3):
        counts = StatevectorSampler().run([full_adder(a, b, c)], shots=50).result()[0].data
        got = list(counts.values())[0].get_counts()
        total = a + b + c
        if got != {f"{total // 2}{total % 2}": 50}:
            return False
    return True

check("full adder is correct for all 8 inputs", _fa)

<details>
<summary>Solution</summary>

```python
def full_adder(a_val, b_val, c_val):
    q = QuantumRegister(5, "q")          # a, b, c, sum, carry = q0..q4
    out = ClassicalRegister(2, "out")
    qc = QuantumCircuit(q, out)
    for i, v in enumerate((a_val, b_val, c_val)):
        if v: qc.x(i)
    qc.cx(0, 3); qc.cx(1, 3); qc.cx(2, 3)                  # sum = a xor b xor c
    qc.ccx(0, 1, 4); qc.ccx(1, 2, 4); qc.ccx(0, 2, 4)      # carry = ab xor bc xor ac
    qc.measure(3, out[0]); qc.measure(4, out[1])
    return qc
```
</details>

**Stretch.** Chain two half adders and an OR (or XOR) to build the full adder from smaller pieces, as a classical engineer would. Compare the gate counts after transpiling to `FakeSherbrooke`. Toffoli gates are expensive on hardware, as you will see.

## Where we go next

The chapters above are the building blocks. Over the rest of the week they combine into algorithms.

| Topic | Builds on |
|---|---|
| Deutsch-Jozsa | Superposition and interference (Chapter 7), the oracle idea from Chapter 10 |
| Quantum Fourier Transform (QFT) | Phase gates and interference (Chapters 3, 7) |
| Quantum Phase Estimation (QPE) | QFT and controlled gates |
| Variational Quantum Classifier (VQC) | Parametrised circuits (Chapter 5) |
| Variational Quantum Eigensolver (VQE) | Parametrised circuits and the Estimator (Chapters 4, 5) |
| QAOA | Parametrised circuits and the Sampler |
| Sample-based Quantum Diagonalization (SQD) | The Sampler on real hardware (Chapter 9) plus classical post-processing |

## Quick reference

```python
qc = QuantumCircuit(n)                 # new circuit with n qubits
qc.h(0); qc.x(0); qc.cx(0, 1)          # gates
qc.rx(theta, 0)                        # rotations: rx, ry, rz
qc.measure_all()                       # or qc.measure(qubit, clbit)
qc.draw("mpl")                         # draw

Statevector(qc)                        # exact state of a circuit without measurements
Operator(qc)                           # exact unitary matrix
plot_bloch_multivector(sv)             # Bloch spheres
plot_histogram(counts)                 # histogram

StatevectorSampler, StatevectorEstimator       # exact reference primitives
AerSimulator(noise_model=...)                  # fast simulator, optionally noisy
generate_preset_pass_manager(backend=..., optimization_level=...)   # transpile
QiskitRuntimeService, SamplerV2, EstimatorV2   # real hardware
```

## Further reading

IBM Quantum documentation:
- [Construct circuits](https://quantum.cloud.ibm.com/docs/en/guides/construct-circuits)
- [Introduction to primitives](https://quantum.cloud.ibm.com/docs/en/guides/primitives)
- [Introduction to transpilation](https://quantum.cloud.ibm.com/docs/en/guides/transpile)
- [Build noise models](https://quantum.cloud.ibm.com/docs/en/guides/build-noise-models)
- [Error mitigation and suppression](https://quantum.cloud.ibm.com/docs/en/guides/error-mitigation-and-suppression-techniques)
- [Execution modes](https://quantum.cloud.ibm.com/docs/en/guides/execution-modes)

Free courses on IBM Quantum Learning:
- [Basics of quantum information](https://quantum.cloud.ibm.com/learning/en/courses/basics-of-quantum-information)
- [Fundamentals of quantum algorithms](https://quantum.cloud.ibm.com/learning/en/courses/fundamentals-of-quantum-algorithms)

Textbook for the mathematics: Nielsen and Chuang, *Quantum Computation and Quantum Information*.